In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import multiprocessing

print(f"CPU cores: {multiprocessing.cpu_count()}")
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3):.1f} GB")

CPU cores: 8
RAM: 51.0 GB


In [2]:
!git clone https://github.com/emadonev/nbody_exoplanets.git

Cloning into 'nbody_exoplanets'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 225 (delta 116), reused 174 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 34.30 MiB | 34.88 MiB/s, done.
Resolving deltas: 100% (116/116), done.


In [3]:
%cd /content/nbody_exoplanets
!pip install -e .

/content/nbody_exoplanets
Obtaining file:///content/nbody_exoplanets
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 101.4 MB/s eta 0:00:00
  Building editable for nbody-exoplanets (pyproject.toml) ... done
  Created wheel for nbody-exoplanets: filename=nbody_exoplanets-0.1.0-0.editable-py3-none-any.whl size=2902 sha256=b95d41f0ebea509b79b3c8c8d60a295fc8b9f1480e395e501ecd668859910b56
  Stored in directory: /tmp/pip-ephem-wheel-cache-152bk9_d/wheels/16/f8/89/6b9a6201883636c260cc939e404e0ed1dceebf04a7e7300c93
Successfully built nbody-exoplanets
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ER

In [4]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from numba import jit
import scipy
import pandas as pd

In [5]:
planets = pd.read_csv("input/planets_triple.csv")

In [6]:
triples = pd.read_csv('input/final_triple_5.csv')

In [7]:
from integrator import run_system
from integrator.experiments import generate_habitability_experiments

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
output_dir = "/content/drive/MyDrive/nbody_outputs"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


In [ ]:
configs = generate_habitability_experiments(
    triples,
    planets,
    tf=1e5,
    output_dir=output_dir,
)

# Override snapshot cadence: ~10,000 snapshots per run (one every ~10 years)
for cfg in configs:
    cfg["output_every_n"] = max(1, int(cfg["tf"] / cfg["dt"] / 10000))

print(f"Generated {len(configs)} experiment configs")
print(configs[0]['experiment_name'])
print(configs[0]['internal_to_physical'])

Generated 106 experiment configs
Gliese_667_observed_ev0.0_iv000
{'A': 'A', 'B': 'B', 'C': 'C'}


In [10]:
preview_df = pd.DataFrame([
    {
        'system_name': cfg['system_name'],
        'planet_scenario': cfg['planet_scenario'],
        'host_star': cfg['host_star'],
        'system_type': cfg['system_type'],
        'outer_e': cfg['outer_e'],
        'outer_i': cfg['outer_i'],
        'source_outer_e': cfg['source_outer_e'],
        'source_outer_i': cfg['source_outer_i'],
        'source_inner_e': cfg['source_inner_e'],
        'source_inner_i': cfg['source_inner_i'],
        'n_planets': len(cfg['planets']),
        'experiment_name': cfg['experiment_name'],
        'output_file': cfg['output_file'],
    }
    for cfg in configs
])

summary_df = (
    preview_df.groupby(['system_name', 'planet_scenario'], as_index=False)
    .agg(
        n_runs=('experiment_name', 'size'),
        outer_e_values=('outer_e', lambda s: sorted(pd.unique(s))),
        outer_i_values=('outer_i', lambda s: sorted(pd.unique(s))),
        n_planets=('n_planets', 'first'),
    )
    .sort_values(['system_name', 'planet_scenario'])
)

print(f"Total configs: {len(preview_df)}")
display(summary_df)
display(preview_df.head(20))

Total configs: 106


,system_name,planet_scenario,n_runs,outer_e_values,outer_i_values,n_planets
0,94 Ceti,hz_inner,1,[0.26],[104.0],1
1,94 Ceti,hz_mid,1,[0.26],[104.0],1
2,94 Ceti,hz_outer,1,[0.26],[104.0],1
3,Gliese 667,observed,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",5
4,HD 132563,hz_inner,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",1
5,HD 132563,hz_mid,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",1
6,HD 132563,hz_outer,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",1
7,Kepler-444,hz_inner,1,[0.55],[85.4],1
8,Kepler-444,hz_mid,1,[0.55],[85.4],1
9,Kepler-444,hz_outer,1,[0.55],[85.4],1


,system_name,planet_scenario,host_star,system_type,outer_e,outer_i,source_outer_e,source_outer_i,source_inner_e,source_inner_i,n_planets,experiment_name,output_file
0,Gliese 667,observed,C,S(C),0.0,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv000,/content/drive/MyDrive/nbody_outputs/Gliese_66...
1,Gliese 667,observed,C,S(C),0.0,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv030,/content/drive/MyDrive/nbody_outputs/Gliese_66...
2,Gliese 667,observed,C,S(C),0.0,60.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv060,/content/drive/MyDrive/nbody_outputs/Gliese_66...
3,Gliese 667,observed,C,S(C),0.0,90.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv090,/content/drive/MyDrive/nbody_outputs/Gliese_66...
4,Gliese 667,observed,C,S(C),0.2,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv000,/content/drive/MyDrive/nbody_outputs/Gliese_66...
5,Gliese 667,observed,C,S(C),0.2,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv030,/content/drive/MyDrive/nbody_outputs/Gliese_66...
6,Gliese 667,observed,C,S(C),0.2,60.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv060,/content/drive/MyDrive/nbody_outputs/Gliese_66...
7,Gliese 667,observed,C,S(C),0.2,90.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv090,/content/drive/MyDrive/nbody_outputs/Gliese_66...
8,Gliese 667,observed,C,S(C),0.4,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.4_iv000,/content/drive/MyDrive/nbody_outputs/Gliese_66...
9,Gliese 667,observed,C,S(C),0.4,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.4_iv030,/content/drive/MyDrive/nbody_outputs/Gliese_66...


In [11]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

max_workers = min(4, multiprocessing.cpu_count())
batch_size = max_workers * 4
resume_existing = False
test_mode = True
test_limit = 8

if resume_existing:
    jobs = [cfg for cfg in configs if not os.path.exists(cfg['output_file'])]
else:
    jobs = list(configs)

if test_mode:
    jobs = jobs[:test_limit]

print(f"Workers: {max_workers}")
print(f"Batch size: {batch_size}")
print(f"Jobs queued: {len(jobs)}")

Workers: 4
Batch size: 16
Jobs queued: 8


In [12]:
results = []
errors = []

for batch_start in range(0, len(jobs), batch_size):
    batch = jobs[batch_start:batch_start + batch_size]
    batch_no = batch_start // batch_size + 1
    total_batches = int(np.ceil(len(jobs) / batch_size)) if jobs else 0
    print(f"Batch {batch_no}/{total_batches}: running {len(batch)} jobs")

    with ProcessPoolExecutor(
        max_workers=max_workers,
        mp_context=mp.get_context('spawn'),
    ) as pool:
        future_to_cfg = {
            pool.submit(run_system, cfg): cfg
            for cfg in batch
        }

        for future in as_completed(future_to_cfg):
            cfg = future_to_cfg[future]
            name = cfg['experiment_name']
            try:
                output_path = future.result()
                results.append({
                    'experiment_name': name,
                    'system_name': cfg['system_name'],
                    'planet_scenario': cfg['planet_scenario'],
                    'output_file': output_path,
                })
                print(f"OK: {name}")
            except Exception as exc:
                errors.append({
                    'experiment_name': name,
                    'system_name': cfg['system_name'],
                    'planet_scenario': cfg['planet_scenario'],
                    'error': repr(exc),
                })
                print(f"FAIL: {name} -> {exc}")

results_df = pd.DataFrame(results)
errors_df = pd.DataFrame(errors)

print(f"Completed: {len(results_df)}")
print(f"Errors: {len(errors_df)}")
display(results_df.head(20))
display(errors_df.head(20))

Batch 1/1: running 8 jobs
OK: Gliese_667_observed_ev0.0_iv030
OK: Gliese_667_observed_ev0.0_iv000
OK: Gliese_667_observed_ev0.0_iv060
OK: Gliese_667_observed_ev0.0_iv090
OK: Gliese_667_observed_ev0.2_iv000
OK: Gliese_667_observed_ev0.2_iv060
OK: Gliese_667_observed_ev0.2_iv030
OK: Gliese_667_observed_ev0.2_iv090
Completed: 8
Errors: 0


,experiment_name,system_name,planet_scenario,output_file
0,Gliese_667_observed_ev0.0_iv030,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
1,Gliese_667_observed_ev0.0_iv000,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
2,Gliese_667_observed_ev0.0_iv060,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
3,Gliese_667_observed_ev0.0_iv090,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
4,Gliese_667_observed_ev0.2_iv000,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
5,Gliese_667_observed_ev0.2_iv060,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
6,Gliese_667_observed_ev0.2_iv030,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...
7,Gliese_667_observed_ev0.2_iv090,Gliese 667,observed,/content/drive/MyDrive/nbody_outputs/Gliese_66...


""
